# Problema 1 — Regressão linear para produção de energia

## Preparação dos dados

Estimamos a potência **PE (MW)** a partir da temperatura (AT), do vácuo de escape (V), da pressão atmosférica (AP) e da umidade relativa (RH).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 12

In [2]:
df = pd.read_excel('data/CCPP/Folds5x2_pp.xlsx')
df.head()

,AT,V,AP,RH,PE
0,14.96,41.76,1024.07,73.17,463.26
1,25.18,62.96,1020.04,59.08,444.37
2,5.11,39.40,1012.16,92.14,488.56
3,20.86,57.32,1010.24,76.64,446.48
4,10.82,37.50,1009.23,96.62,473.90


In [3]:
print("Valores ausentes por coluna:")
print(df.isnull().sum())

print("\nDuplicatas exatas encontradas:", df.duplicated().sum())

Valores ausentes por coluna:
AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

Duplicatas exatas encontradas: 41


In [4]:
df = df.drop_duplicates(keep='first').reset_index(drop=True)
print(f'Dimensões após a limpeza: {df.shape[0]} observações e {df.shape[1]} variáveis')
print(f'Duplicatas restantes: {df.duplicated().sum()}')

Dimensões após a limpeza: 9527 observações e 5 variáveis
Duplicatas restantes: 0


In [5]:
X = df.drop(columns='PE')
y = df['PE']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print(f'Treino: {X_train.shape[0]} observações ({X_train.shape[0] / len(df) * 100:.1f}%)')
print(f'Teste:  {X_test.shape[0]} observações ({X_test.shape[0] / len(df) * 100:.1f}%)')

Treino: 7621 observações (80.0%)
Teste:  1906 observações (20.0%)


## Tarefa 1 — Correlação dos sensores com PE

Calculamos a correlação de Pearson no treino. O sinal indica o sentido da associação; o módulo indica sua intensidade linear.

In [6]:
dados_treino = pd.concat([X_train, y_train], axis=1)

In [7]:
correlacoes = dados_treino.corr()
display(correlacoes['PE'].drop('PE').round(3).to_frame('Correlação com PE'))

,Correlação com PE
AT,-0.948
V,-0.868
AP,0.519
RH,0.386


| Sensor | Correlação com PE | Leitura |
|---|---:|---|
| AT | −0,948 | Maior temperatura se associa a menor potência. |
| V | −0,868 | Valores maiores de vácuo de escape se associam a menor potência. |
| AP | +0,519 | Maior pressão se associa a maior potência. |
| RH | +0,386 | Maior umidade se associa a maior potência. |

**AT e V apresentam as associações mais fortes**, pois têm as maiores correlações em módulo. Essa comparação é marginal: não mede o efeito isolado de cada sensor nem demonstra causalidade.

## Tarefa 2 — Correlação entre os sensores

A matriz permite verificar se os preditores carregam informações semelhantes.

In [8]:
display(correlacoes.loc[X.columns, X.columns].round(3))

,AT,V,AP,RH
AT,1.000,0.840,-0.508,-0.540
V,0.840,1.000,-0.413,-0.305
AP,-0.508,-0.413,1.000,0.094
RH,-0.540,-0.305,0.094,1.000


**AT–V é o par mais correlacionado: r = +0,840.**

Essa associação sugere multicolinearidade: parte da informação de AT também está presente em V. Isso pode tornar os coeficientes individuais menos estáveis, mesmo quando o modelo produz boas previsões.

Os demais pares têm correlações menores em módulo; não há correlação perfeita entre as entradas.

Para previsão, a correlação AT–V não exige excluir automaticamente um sensor: mantemos os quatro, como solicitado. Para interpretar seus efeitos individuais, porém, os coeficientes devem ser lidos em conjunto.


## Tarefa 3 — Regressão linear múltipla

Usamos os quatro sensores para estimar PE. A divisão 80/20 já foi realizada na preparação; agora padronizamos as entradas e ajustamos o modelo somente com o treino.

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# O escalonador é ajustado apenas no treino, as mesmas estatísticas são usadas no teste.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

modelo = LinearRegression().fit(X_train_scaled, y_train)

In [10]:
y_train_pred = modelo.predict(X_train_scaled)
y_test_pred = modelo.predict(X_test_scaled)


O modelo combina linearmente as quatro entradas:

$$
\widehat{PE} = \beta_0 + \beta_{AT}AT + \beta_VV + \beta_{AP}AP + \beta_{RH}RH
$$

Essa expressão representa a previsão nas unidades originais. No código, o ajuste é feito sobre as entradas padronizadas.

- **Treino:** 7.621 observações usadas para estimar a escala e os coeficientes.
- **Teste:** 1.906 observações transformadas com a mesma escala, sem novo ajuste.

## Tarefa 4 — Avaliação no teste

Comparamos as previsões com os valores observados usando R², RMSE e MAE. Os erros são expressos em MW, a unidade de PE.

In [11]:
def avaliar_modelo(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return r2, rmse, mae

r2_tr, rmse_tr, mae_tr = avaliar_modelo(y_train, y_train_pred)
r2_te, rmse_te, mae_te = avaliar_modelo(y_test, y_test_pred)

tabela_metricas = pd.DataFrame({
    'Métrica': ['Coeficiente de Determinação (R²)', 'Raiz do Erro Quadrático Médio (RMSE)', 'Erro Absoluto Médio (MAE)'],
    'Treino': [f"{r2_tr:.4f}", f"{rmse_tr:.4f} MW", f"{mae_tr:.4f} MW"],
    'Teste':  [f"{r2_te:.4f}", f"{rmse_te:.4f} MW", f"{mae_te:.4f} MW"]
})
display(tabela_metricas)

,Métrica,Treino,Teste
0,Coeficiente de Determinação (R²),0.9284,0.9284
1,Raiz do Erro Quadrático Médio (RMSE),4.5512 MW,4.5875 MW
2,Erro Absoluto Médio (MAE),3.6252 MW,3.6215 MW


### Desempenho obtido

| Métrica | Valor no teste | Interpretação |
|---|---:|---|
| R² | **0,9284** | Explica cerca de 92,84% da variação de PE neste teste. |
| RMSE | **4,59 MW** | Resume o erro dando maior peso aos desvios grandes. |
| MAE | **3,62 MW** | A previsão se afasta do valor real, em média, 3,62 MW. |

### Adequação do modelo

O modelo é uma referência adequada para condições semelhantes às observadas. Com um MAE de 3,62 MW, a previsão se afasta do valor real, em média, 3,62 MW. 

O desempenho sustenta o modelo como referência de previsão nesta base. Para usá-lo em produção, seria necessário confrontar os erros com a tolerância da usina e verificar seu comportamento em períodos futuros.
